# Algebra Quiz Generation Pipeline

This notebook generates quizzes for all algebra topics using the benchmark prompt system.

## Topics Covered:
- Linear Equations
- Quadratic Equations  
- Systems of Equations
- Polynomials
- Functions
- Exponents and Radicals
- Rational Expressions
- Inequalities

In [ ]:
# Import required modules
import sys
import os
from pathlib import Path
import json
from datetime import datetime

# Add project root to path
project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

from prompts.math.algebra import AlgebraBenchmark
from prompts.benchmark_manager import BenchmarkManager

## Setup Configuration

In [ ]:
# Configuration
QUESTIONS_PER_TOPIC = 5  # Number of questions to generate per topic
MODEL_NAME = "gemini-2.5-flash-lite"  # Change this to test different models
OUTPUT_DIR = Path(f"../../generated_quizzes/{MODEL_NAME}/math")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Initialize algebra benchmark
algebra = AlgebraBenchmark()
manager = BenchmarkManager()

print(f"✓ Algebra benchmark initialized")
print(f"✓ Available topics: {algebra.list_topics()}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ Target model: {MODEL_NAME}")

## Generate Quizzes for All Topics

In [ ]:
# Generate quizzes for all algebra topics
generated_quizzes = {}

print("🚀 Starting algebra quiz generation...")
print("=" * 50)

# Using topic-specific methods
topics_and_methods = [
    ("Linear Equations", algebra.linear_equations),
    ("Quadratic Equations", algebra.quadratic_equations),
    ("Systems of Equations", algebra.systems_of_equations),
    ("Polynomials", algebra.polynomials),
    ("Functions", algebra.functions),
    ("Exponents and Radicals", algebra.exponents_and_radicals),
    ("Rational Expressions", algebra.rational_expressions),
    ("Inequalities", algebra.inequalities)
]

for topic_name, method in topics_and_methods:
    print(f"\n📝 Generating {topic_name} quiz...")
    
    try:
        # Generate benchmark prompt
        benchmark_prompt = method(QUESTIONS_PER_TOPIC)
        
        # Store the generated prompt
        generated_quizzes[topic_name] = {
            "benchmark_prompt": benchmark_prompt,
            "user_prompt": benchmark_prompt.user_prompt,
            "system_prompt": benchmark_prompt.system_prompt,
            "topic": benchmark_prompt.topic,
            "question_count": benchmark_prompt.question_count,
            "difficulty": benchmark_prompt.expected_difficulty,
            "quality_metrics": benchmark_prompt.quality_metrics,
            "generated_at": datetime.now().isoformat()
        }
        
        print(f"   ✓ Generated {benchmark_prompt.question_count} questions")
        print(f"   ✓ Difficulty: {benchmark_prompt.expected_difficulty}")
        
    except Exception as e:
        print(f"   ❌ Error generating {topic_name}: {e}")
        generated_quizzes[topic_name] = {"error": str(e)}

print("\n" + "=" * 50)
print(f"✅ Completed! Generated {len([q for q in generated_quizzes.values() if 'error' not in q])} successful quizzes")

## Save Generated Quizzes

In [ ]:
# Save generated quizzes to JSON files
print("💾 Saving generated quizzes...")

for topic_name, quiz_data in generated_quizzes.items():
    if "error" not in quiz_data:
        # Create filename
        filename = f"algebra_{topic_name.lower().replace(' ', '_')}.json"
        filepath = OUTPUT_DIR / filename
        
        # Save to file
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(quiz_data, f, indent=2, ensure_ascii=False)
        
        print(f"   ✓ Saved {filename}")
    else:
        print(f"   ❌ Skipped {topic_name} (had errors)")

print(f"\n📁 All quizzes saved to: {OUTPUT_DIR}")

## Generate Comprehensive Algebra Benchmark

In [ ]:
# Generate a comprehensive benchmark covering all algebra topics
print("🎯 Generating comprehensive algebra benchmark...")

try:
    comprehensive_suite = manager.generate_subject_benchmark('algebra', questions_per_topic=3)
    
    # Save comprehensive benchmark
    comprehensive_path = OUTPUT_DIR / "algebra_comprehensive_benchmark.json"
    
    export_data = {
        "name": comprehensive_suite.name,
        "metadata": comprehensive_suite.metadata,
        "prompts": [
            {
                "subject": prompt.subject,
                "topic": prompt.topic,
                "expected_difficulty": prompt.expected_difficulty,
                "question_count": prompt.question_count,
                "quality_metrics": prompt.quality_metrics,
                "user_prompt": prompt.user_prompt,
                "system_prompt": prompt.system_prompt
            }
            for prompt in comprehensive_suite.prompts
        ]
    }
    
    with open(comprehensive_path, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Generated comprehensive benchmark with {len(comprehensive_suite.prompts)} topic prompts")
    print(f"✓ Total questions: {comprehensive_suite.metadata['total_questions']}")
    print(f"✓ Saved to: {comprehensive_path}")
    
except Exception as e:
    print(f"❌ Error generating comprehensive benchmark: {e}")

## Summary Report

In [ ]:
# Generate summary report
print("📊 Algebra Quiz Generation Summary")
print("=" * 40)

successful_quizzes = [q for q in generated_quizzes.values() if 'error' not in q]
failed_quizzes = [q for q in generated_quizzes.values() if 'error' in q]

total_questions = sum(q['question_count'] for q in successful_quizzes)
total_topics = len(successful_quizzes)

print(f"📈 Successful quizzes: {total_topics}/8 topics")
print(f"📝 Total questions generated: {total_questions}")
print(f"📁 Output location: {OUTPUT_DIR}")
print(f"⏰ Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

if failed_quizzes:
    print(f"❌ Failed topics: {len(failed_quizzes)}")
    for topic, data in generated_quizzes.items():
        if 'error' in data:
            print(f"   - {topic}: {data['error']}")
else:
    print("✅ All topics generated successfully!")

print("\n🎉 Algebra quiz generation completed!")